In [42]:
import json
from pathlib import Path

from sqlalchemy.ext.asyncio import create_async_engine

from agent import query_llm, raw_query_llm
from sql_layer import DEFAULT_MODEL, build_messages, query_to_sqllite, sql_validator

In [43]:
db_dir = Path.cwd().parent / "data" / "db"
db_files = sorted(db_dir.glob("construction*.db"))

if not db_files:
    raise FileNotFoundError("Database file matching 'construction*.db' was not found")

db_path = db_files[0]

async_engine = create_async_engine(f"sqlite+aiosqlite:///{db_path}")

In [44]:
user_query = "Мне нужна инофрмация какие работы сейчас выполняет подрядчик Строймонтаж, а также их % готовности."

In [45]:
async def sql_layer(**kwargs) -> str | list[tuple]:
    async_engine = create_async_engine(f"sqlite+aiosqlite:///{db_path}")
    user_query = kwargs["user_question"]
    messages = await build_messages(user_query, async_engine)

    for i in range(3):
        sql_query = await sql_validator(messages)
        if sql_query["status"] == "ok":
            break
        if i == 2:
            return "Запрос не прошел валидацию, проверьте корректность запроса или попробуйте его переформулировать."

    db_result = await query_to_sqllite(sql_query["sql"])

    return db_result

In [46]:
TOOL_MAPPING = {"sql_layer": sql_layer}

In [47]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "sql_layer",
            "description": (
                "Принимает вопрос на естественном языке, генерирует и валидирует "
                "SQL-запрос к строительной БД (VIEW v_construction_data). "
                "Возвращает готовый SQL, статус выполнения и (опционально) строки результата. "
                "Используется для аналитики: объёмы работ, подрядчики, бюджеты, прогресс, "
                "сравнение план/факт по городам, объектам и типам работ."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "user_question": {
                        "type": "string",
                        "description": (
                            "Вопрос пользователя на естественном языке, например: "
                            "'Какой процент выполнения по каждому объекту?', "
                            "'Покажи объёмы работ подрядчика ООО Строй'."
                        ),
                    },
                    "model_name": {
                        "type": "string",
                        "description": (
                            "Идентификатор модели на OpenRouter "
                            "(default: 'meta-llama/llama-3.3-70b-instruct')."
                        ),
                    },
                },
                "required": ["user_question"],
            },
        },
    }
]


In [48]:
messages = [
    {
        "role": "system",
        "content": "Ты BI аналитик, который на основе запроса делает поиск релевантной информации и строит отчет саммари. Для поиска информации используй инструменты.",
    },
    {
        "role": "user",
        "content": user_query,
    },
]

result = await raw_query_llm(messages, DEFAULT_MODEL, tools=tools)

In [49]:
if result.choices[0].message.tool_calls:
    func_ids = []
    func_arguments = []
    func_names = []
    for tool in result.choices[0].message.tool_calls:
        func_ids.append(tool.id)
        func_arguments.append(json.loads(tool.function.arguments))
        func_names.append(tool.function.name)

In [50]:
if func_ids:
    for i, _id in enumerate(func_ids):
        messages.append(
            {
                "role": "assistant",
                "content": None,
                "tool_calls": [
                    {
                        "id": _id,
                        "type": "function",
                        "function": {
                            "name": func_names[i],
                            "arguments": str(func_arguments[i]),
                        },
                    }
                ],
            }
        )

        try:
            func_result = await TOOL_MAPPING[func_names[i]](**func_arguments[i])

            messages.append(
                {
                    "role": "tool",
                    "content": str(func_result),
                }
            )
        except Exception as err:
            messages.append(
                {
                    "role": "user",
                    "content": f"Вызов функции: {func_names[i]}; с аргументами: {func_arguments[i]}; вернул ошибку: {err}.\nПроанализируй в чем проблема и исправь запрос.",
                }
            )

In [51]:
messages.extend(
    [
        {
            "role": "user",
            "content": "Сформируй краткий ответ на запрос пользователя, на основе полученной ранее информации.",
        }
    ]
)

In [52]:
messages

[{'role': 'system',
  'content': 'Ты BI аналитик, который на основе запроса делает поиск релевантной информации и строит отчет саммари. Для поиска информации используй инструменты.'},
 {'role': 'user',
  'content': 'Мне нужна инофрмация какие работы сейчас выполняет подрядчик Строймонтаж, а также их % готовности.'},
 {'role': 'assistant',
  'content': None,
  'tool_calls': [{'id': 'chatcmpl-tool-61b1118de81a4f92855289e14169eae5',
    'type': 'function',
    'function': {'name': 'sql_layer',
     'arguments': "{'user_question': 'Какие работы сейчас выполняет подрядчик Строймонтаж и их процент готовности?'}"}}]},
 {'role': 'tool',
  'content': "[('work_type', 'unit', 'sum_fact_vol', 'sum_plan_vol', 'процент_выполнения'), ('Внешняя отделка', 'кв.м', 524.36, 1995.56, 26.28), ('Внутренняя отделка', 'комплект', 3465.1, 4624.66, 74.93), ('Водоснабжение', 'м²', 1012.99, 1235.82, 81.97), ('Возведение стен', 'км', 781.62, 1106.81, 70.62), ('Земляные работы', 'тонн', 107.78, 233.12, 46.23), ('Мон

In [53]:
final_answer = await query_llm(messages=messages, model_name=DEFAULT_MODEL)

In [59]:
print(final_answer)

Подрядчик Строймонтаж в настоящее время выполняет следующие работы:

1. Внешняя отделка (26,28% готовности)
2. Внутренняя отделка (74,93% готовности)
3. Водоснабжение (81,97% готовности)
4. Возведение стен (70,62% готовности)
5. Земляные работы (46,23% готовности)
6. Монтаж перекрытий (61,47% готовности)
7. Окраска (22,36% готовности)
8. Отопление (100% готовности)
9. Санитарно-техническая подготовка (17,16% готовности)
10. Установка окон (61,68% готовности)

Отопление полностью завершено, а работы по внутренней отделке и водоснабжению находятся на продвинутой стадии.


### RAW

In [54]:
messages = await build_messages(user_query, async_engine)
messages

[{'role': 'system',
  'content': "Ты преобразуешь запросы на естественном языке в один корректный SQL-запрос для SQLite.\n\nЗАДАЧА\n\nПострой ровно один SQL-запрос, используя только VIEW и допустимые значения ниже.\nЕсли построить запрос невозможно — верни ровно: Невозможно ответить\n\n\nСХЕМА VIEW\n\nv_construction_data: object_id, object_name, city, object_budget, contractor_id, contractor_name, work_id, work_type, unit, work_plan_vol, work_fact_vol, work_labor_plan, work_labor_fact, progress_id, progress_date, progress_plan_vol, progress_fact_vol, progress_labor_plan, progress_labor_fact, progress_completion_pct\n\nИспользуй ТОЛЬКО v_construction_data. НЕ обращайся напрямую к таблицам objects, contractors, works, progress.\n\nКолонки VIEW:\n  Объект:     object_id, object_name, city, object_budget\n  Подрядчик:  contractor_id, contractor_name\n  Работа:     work_id, work_type, unit, work_plan_vol, work_fact_vol, work_labor_plan, work_labor_fact\n  Прогресс:   progress_id, progress_d

In [55]:
for i in range(3):
    sql_query = await sql_validator(messages)
    if sql_query["status"] == "ok":
        break

sql_query

{'sql': "SELECT\n  work_type,\n  unit,\n  ROUND(SUM(progress_plan_vol), 2) AS sum_plan_vol,\n  ROUND(SUM(progress_fact_vol), 2) AS sum_fact_vol,\n  ROUND(SUM(progress_fact_vol) * 100.0 / SUM(progress_plan_vol), 2) AS процент_выполнения\nFROM v_construction_data\nWHERE\n  contractor_name = 'АО Строймонтаж'\nGROUP BY\n  work_type,\n  unit\nORDER BY\n  work_type ASC,\n  unit ASC",
 'rows': None,
 'answer': None,
 'status': 'ok'}

In [56]:
print(sql_query["sql"])

SELECT
  work_type,
  unit,
  ROUND(SUM(progress_plan_vol), 2) AS sum_plan_vol,
  ROUND(SUM(progress_fact_vol), 2) AS sum_fact_vol,
  ROUND(SUM(progress_fact_vol) * 100.0 / SUM(progress_plan_vol), 2) AS процент_выполнения
FROM v_construction_data
WHERE
  contractor_name = 'АО Строймонтаж'
GROUP BY
  work_type,
  unit
ORDER BY
  work_type ASC,
  unit ASC


In [57]:
db_result = await query_to_sqllite(sql_query["sql"])
db_result

[('work_type', 'unit', 'sum_plan_vol', 'sum_fact_vol', 'процент_выполнения'),
 ('Внешняя отделка', 'кв.м', 1995.56, 524.36, 26.28),
 ('Внутренняя отделка', 'комплект', 4624.66, 3465.1, 74.93),
 ('Водоснабжение', 'м²', 1235.82, 1012.99, 81.97),
 ('Возведение стен', 'км', 1106.81, 781.62, 70.62),
 ('Земляные работы', 'тонн', 233.12, 107.78, 46.23),
 ('Монтаж перекрытий', 'м³', 2854.48, 1754.58, 61.47),
 ('Окраска', 'км', 547.69, 122.48, 22.36),
 ('Отопление', 'комплект', 514.58, 514.58, 100.0),
 ('Санитарно-техническая подготовка', 'тонн', 2057.67, 353.13, 17.16),
 ('Установка окон', 'м²', 1529.47, 943.36, 61.68)]